# Two-dimensional assignments for different cost exponents

This notebook generates `fig:matching-2d-cost-exponent`.  It defines the canonical two-dimensional point-cloud pair reused by several later figures: a central Gaussian source and a target made of three Gaussian modes around it.  For the same clouds, we solve
$$
\min_{\sigma\in\operatorname{Perm}(n)} \frac1n\sum_{i=1}^n \|x_i-y_{\sigma(i)}\|^p
$$
for $p=1,2,6$.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np
import ot

from figure_style import (
    VIOLET,
    canonical_matching_clouds,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()


## Canonical source and target clouds

The source is deliberately compact and central.  The target has three separated modes with unequal spreads, which creates competing assignments while remaining easy to read.

In [ ]:
alpha, beta, beta_labels = canonical_matching_clouds(seed=2027, n_source=36)

all_points = np.vstack([alpha, beta])
xlim, ylim = padded_limits(all_points, pad=0.18)
D = np.linalg.norm(alpha[:, None, :] - beta[None, :, :], axis=2)
a = np.ones(len(alpha)) / len(alpha)
b = np.ones(len(beta)) / len(beta)


## Assignments and export

For equal weights and equal cardinality, the optimal transport plan is a permutation matrix.  We export one axis-free panel per exponent so LaTeX can assemble the comparison.

In [ ]:
fig_name = "matching-2d-cost-exponent"
out = figure_dir(fig_name)

for p, filename in [(1, "p1.pdf"), (2, "p2.pdf"), (6, "p6.pdf")]:
    plan = ot.emd(a, b, D ** p, numItermax=200000)
    pairs = [(i, j, float(plan[i, j])) for i in range(plan.shape[0]) for j in range(plan.shape[1]) if plan[i, j] > 1e-10]

    fig, ax = plt.subplots(figsize=(2.25, 2.25))
    draw_transport_segments(ax, alpha, beta, pairs, color=VIOLET, min_width=0.36, max_width=0.36, alpha_scale=0.50, zorder=1)
    draw_point_clouds(ax, alpha, beta, base_size=13.0, zorder=3)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    remove_axes(ax)
    save_pdf(fig, out / filename, pad_inches=0.075)
    plt.close(fig)
